# 02 - Silver: Tratamento e Modelagem
Pipeline de limpeza em etapas, aplicado sobre a camada Bronze:
1. Tipagem de colunas
2. Deduplicação
3. Tratamento de nulos/inválidos
4. Padronização de texto
5. Regras de negócio (consistência de datas, valores)
6. Validação final

In [0]:
#Setup
from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG = "olist_project"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

## Etapa 0: Diagnóstico
Antes de definir as regras de limpeza, olhamos para nulos, tipos atuais e valores distintos
que possam indicar inconsistência.

In [0]:
def diagnosticar(tabela):
    df = spark.table(f"{BRONZE_SCHEMA}.{tabela}")
    total = df.count()
    #print(f"\n=== {tabela} ({total} linhas) ===")
    print(f"\n---------------- Tabela: {tabela} ({total} linhas) ----------------")

    nulos_wide = df.select([
        F.round(
            F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)) / total * 100,
            2
        ).alias(c)
        for c in df.columns if not c.startswith("_")
    ])

    # Transforma a única linha "larga" em uma lista de (coluna, percentual)
    linha = nulos_wide.collect()[0].asDict()
    resultado = sorted(linha.items(), key=lambda x: x[1], reverse=True)

    # Cabeçalho
    print(f"{'CAMPO':40s} {'PERCENTUAL DE NULOS':>10s}")

    for coluna, percentual in resultado:
        print(f"{coluna:40s} {percentual:6.2f}%")

    #print("-" * 50) 

In [0]:
#Rodar o diagnóstico nas tabelas principais
for tabela in ["orders", "customers", "order_items", "payments", "reviews", "products", "sellers", "geolocation", "category_translation"]:
    diagnosticar(tabela)

%md
## Etapa 1: Tipagem
Bronze é tudo string. Aqui convertemos cada coluna para o tipo correto, usando `try_cast`
para nunca quebrar o pipeline — valores malformados viram NULL e são contabilizados.

In [0]:
# ==========================================
# Datas / Timestamps
# ==========================================

Orders_Date = {
    "order_purchase_timestamp": "timestamp",
    "order_approved_at": "timestamp",
    "order_delivered_carrier_date": "timestamp",
    "order_delivered_customer_date": "timestamp",
    "order_estimated_delivery_date": "timestamp",
}

Items_Date = {
    "shipping_limit_date": "timestamp",
}

Reviews_Date = {
    "review_creation_date": "timestamp",
    "review_answer_timestamp": "timestamp",
}


# ==========================================
# Números (inteiros, decimais, valores monetários)
# ==========================================

Items_Number = {
    "price": "decimal(10,2)",
    "freight_value": "decimal(10,2)",
}

Reviews_Number = {
    "review_score": "int",
}

Payments_Number = {
    "payment_value": "decimal(10,2)",
    "payment_installments": "int",
}

Products_Number = {
    "product_weight_g": "double",
    "product_length_cm": "double",
    "product_height_cm": "double",
    "product_width_cm": "double",
    "product_name_lenght": "int",          # nome mantido igual ao dataset original (erro de digitação no Kaggle)
    "product_description_lenght": "int",
    "product_photos_qty": "int",
}

Geolocation_Number = {
    "geolocation_lat": "double",
    "geolocation_lng": "double",
}


# ==========================================
# Strings / Texto (nomes, categorias, endereços)
# ==========================================

# Nesse dataset, colunas de texto já vêm corretas como string na Bronze,
# então normalmente não precisam de cast — só entram aqui se precisar
# forçar/confirmar o tipo explicitamente.

Products_String = {
    "product_category_name": "string",
}

CategoryTranslation_String = {
    "product_category_name": "string",
    "product_category_name_english": "string",
}

Geolocation_String = {
    "geolocation_city": "string",
    "geolocation_state": "string",
}


# ==========================================
# Chaves (IDs) — mantidas como string, nunca numéricas
# ==========================================

Chaves = [
    "order_id", "customer_id", "product_id", "seller_id",
    "review_id", "order_item_id", "customer_unique_id",
    "customer_zip_code_prefix", "seller_zip_code_prefix",
    "geolocation_zip_code_prefix",
]

In [0]:
#Função genérica de cast com contagem de falhas 
def aplicar_tipos(df, tabela, mapa_tipos):
    """mapa_tipos: dict {coluna: tipo_spark_sql}"""
    for coluna, tipo in mapa_tipos.items():
        antes_nulos = df.filter(F.col(coluna).isNull()).count()
        df = df.withColumn(coluna, F.expr(f"try_cast({coluna} as {tipo})"))
        depois_nulos = df.filter(F.col(coluna).isNull()).count()
        novos_nulos = depois_nulos - antes_nulos
        if novos_nulos > 0:
            print(f"⚠️  {tabela}.{coluna}: {novos_nulos} valores não puderam ser convertidos pra {tipo}")
    return df

In [0]:
orders = aplicar_tipos(
    spark.table(f"{BRONZE_SCHEMA}.orders"), "orders", Orders_Date
)

order_items = aplicar_tipos(
    spark.table(f"{BRONZE_SCHEMA}.order_items"),
    "order_items",
    {**Items_Date, **Items_Number}
)

reviews = aplicar_tipos(
    spark.table(f"{BRONZE_SCHEMA}.reviews"),
    "reviews",
    {**Reviews_Date, **Reviews_Number}
)

payments = aplicar_tipos(
    spark.table(f"{BRONZE_SCHEMA}.payments"), "payments", Payments_Number
)

customers = spark.table(f"{BRONZE_SCHEMA}.customers")
sellers = spark.table(f"{BRONZE_SCHEMA}.sellers")

products = aplicar_tipos(
    spark.table(f"{BRONZE_SCHEMA}.products"),
    "products",
    {**Products_Number, **Products_String}
)

geolocation_tipada = aplicar_tipos(
    spark.table(f"{BRONZE_SCHEMA}.geolocation"),
    "geolocation",
    {**Geolocation_Number, **Geolocation_String}
)

geolocation = (geolocation_tipada
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("latitude"),
        F.avg("geolocation_lng").alias("longitude"),
        F.first("geolocation_city").alias("cidade"),
        F.first("geolocation_state").alias("estado"),
    )
    .withColumn("cidade", F.lower(F.trim(F.col("cidade"))))
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
)

category_translation = (spark.table(f"{BRONZE_SCHEMA}.category_translation")
    .withColumn("product_category_name", F.lower(F.trim(F.col("product_category_name"))))
    .withColumn("product_category_name_english", F.lower(F.trim(F.col("product_category_name_english"))))
    .dropDuplicates(["product_category_name"])
)

print("✅ Tipagem/tratamento inicial concluído para todas as tabelas")

## Etapa 2: Deduplicação
Remove linhas duplicadas pela chave primária de cada entidade, registrando quantas foram removidas.

In [0]:
#Função genérica
def deduplicar(df, tabela, chave):
    antes = df.count()
    df = df.dropDuplicates([chave])
    depois = df.count()
    removidas = antes - depois
    if removidas > 0:
        print(f"🔁 {tabela}: {removidas} duplicatas removidas (chave={chave})")
    return df

In [0]:
#Aplicar 
orders = deduplicar(orders, "orders", "order_id")
customers = deduplicar(customers, "customers", "customer_id")
products = deduplicar(products, "products", "product_id")
sellers = deduplicar(sellers, "sellers", "seller_id")
reviews = deduplicar(reviews, "reviews", "review_id")
order_items = order_items.dropDuplicates(["order_id", "order_item_id"])  # chave composta

## Etapa 3: Nulos e valores inválidos
Regras específicas por coluna: remover linha, preencher com valor padrão, ou apenas sinalizar.

In [0]:
#Exemplos de regras

# order_id nulo é inaceitável -> remove a linha
orders = orders.filter(F.col("order_id").isNotNull())

# category_name nulo -> categoriza como "nao_informado" em vez de descartar o produto
products = products.withColumn(
    "product_category_name",
    F.coalesce(F.col("product_category_name"), F.lit("nao_informado"))
)

# review_score fora do intervalo 1-5 -> vira nulo (já tratado pelo try_cast, aqui só validamos)
reviews_invalidas = reviews.filter(~F.col("review_score").between(1, 5))
print("Reviews com score fora do intervalo (após cast):", reviews_invalidas.count())

## Etapa 4: Padronização
Trim, caixa baixa, e normalização de campos de texto usados como chave de agrupamento
(estado, cidade, categoria) — evita "SP" vs "sp " gerarem grupos diferentes no Gold.

In [0]:
customers = (customers
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
    .withColumn("customer_city", F.lower(F.trim(F.col("customer_city"))))
)

sellers = (sellers
    .withColumn("seller_state", F.upper(F.trim(F.col("seller_state"))))
    .withColumn("seller_city", F.lower(F.trim(F.col("seller_city"))))
)

products = products.withColumn(
    "product_category_name", F.lower(F.trim(F.col("product_category_name")))
)

## Etapa 5: Regras de negócio
Consistência temporal: data de entrega não pode ser anterior à compra. Sinalizamos
em vez de descartar, pra não perder pedidos por um possível erro pontual de origem.

In [0]:
orders = orders.withColumn(
    "flag_data_inconsistente",
    F.when(
        F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"),
        True
    ).otherwise(False)
)

qtd_inconsistentes = orders.filter(F.col("flag_data_inconsistente")).count()
print(f"⚠️  {qtd_inconsistentes} pedidos com data de entrega anterior à compra (sinalizados, não removidos)")

## Etapa 6: Gravação

In [0]:
tabelas_tratadas = {
    "orders": orders, "customers": customers, "order_items": order_items,
    "payments": payments, "reviews": reviews, "products": products,
    "sellers": sellers, "geolocation": geolocation,
    "category_translation": category_translation,
}

for nome, df in tabelas_tratadas.items():
    df.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_SCHEMA}.{nome}")
    print(f"✅ silver.{nome}: {df.count()} linhas")

## Etapa 7: Validação
Chave única, integridade referencial entre tabelas, e intervalos válidos.

In [0]:
#Funções de checagem (reaproveitando o padrão do notebook Bronze) 
from pyspark.sql import Row
from datetime import datetime

# def registrar_checagem(camada, tabela, checagem, status, detalhe=""):
#     row = Row(camada=camada, tabela=tabela, checagem=checagem,
#               status=status, detalhe=detalhe, timestamp=datetime.now())
#     spark.createDataFrame([row]).write.format("delta").mode("append") \
#         .saveAsTable(f"{GOLD_SCHEMA}.quality_log")
#     simbolo = "✅" if status == "PASSOU" else "❌"
#     print(f"{simbolo} [{camada}.{tabela}] {checagem} — {detalhe}")

def checar_chave_unica(tabela, coluna):
    df = spark.table(f"{SILVER_SCHEMA}.{tabela}")
    total, distintos = df.count(), df.select(coluna).distinct().count()
    status = "PASSOU" if total == distintos else "FALHOU"
    #registrar_checagem("silver", tabela, f"chave_unica_{coluna}", status, f"{total} vs {distintos}")

# def checar_integridade_referencial(filho, fk, pai, pk):
#     df_filho = spark.table(f"{SILVER_SCHEMA}.{filho}")
#     df_pai = spark.table(f"{SILVER_SCHEMA}.{pai}")
#     orfaos = df_filho.join(df_pai, df_filho[fk] == df_pai[pk], "left_anti").count()
#     status = "PASSOU" if orfaos == 0 else "FALHOU"
#     registrar_checagem("silver", filho, f"integridade_{fk}_para_{pai}", status, f"{orfaos} órfãos")

In [0]:
# Rodar as checagens

# Chaves únicas
checar_chave_unica("orders", "order_id")
checar_chave_unica("customers", "customer_id")
checar_chave_unica("products", "product_id")
checar_chave_unica("reviews", "review_id")
checar_chave_unica("sellers", "seller_id")
checar_chave_unica("geolocation", "geolocation_zip_code_prefix")
checar_chave_unica("category_translation", "product_category_name")

# # Integridade referencial
# checar_integridade_referencial("order_items", "order_id", "orders", "order_id")
# checar_integridade_referencial("orders", "customer_id", "customers", "customer_id")
# checar_integridade_referencial("order_items", "product_id", "products", "product_id")
# checar_integridade_referencial("order_items", "seller_id", "sellers", "seller_id")
# checar_integridade_referencial("payments", "order_id", "orders", "order_id")
# checar_integridade_referencial("reviews", "order_id", "orders", "order_id")

In [0]:
display(spark.sql("""
    SELECT table_schema, table_name, column_name, data_type
    FROM olist_project.information_schema.columns
    WHERE table_schema = 'silver'
    ORDER BY table_name, ordinal_position
"""))

In [0]:
# Contagem de PASSOU/FALHOU por tabela
display(
    spark.table(f"{GOLD_SCHEMA}.quality_log")
    .filter("camada = 'silver'")
    .groupBy("tabela", "status")
    .count()
    .orderBy("tabela")
)